In [0]:
dbutils.widgets.text("catalog", "global_airline", "Catalog Name")
catalog = dbutils.widgets.get("catalog")

In [0]:
silver_passenger_df = spark.read.table(f"{catalog}.silver.passenger")
silver_passenger_df.write.mode("overwrite").saveAsTable(f"{catalog}.gold.dim_passenger")

In [0]:
silver_route_df = spark.read.table(f"{catalog}.silver.route")
silver_route_df.write.mode("overwrite").saveAsTable(f"{catalog}.gold.dim_route")

In [0]:
silver_airline_df = spark.read.table(f"{catalog}.silver.airline")
silver_airline_df.write.mode("overwrite").saveAsTable(f"{catalog}.gold.dim_airline")

In [0]:
silver_nation_df = spark.read.table(f"{catalog}.silver.nation")
silver_region_df = spark.read.table(f"{catalog}.silver.region")

dim_geography_df = silver_nation_df.join(
    silver_region_df,
    silver_nation_df["n_regionkey"] == silver_region_df["r_regionkey"],
    "left"
).select(
    *silver_nation_df.columns,
    silver_region_df["r_name"],
)

dim_geography_df.write.mode("overwrite").saveAsTable(f"{catalog}.gold.dim_geography")

In [0]:
silver_flight_segment_df = spark.read.table(f"{catalog}.silver.flight_booking")
silver_flight_segment_df.write.mode("overwrite").saveAsTable(f"{catalog}.gold.fact_bookings")

In [0]:
# Annual booking KPI view.
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.gold.vw_annual_booking_kpis AS
SELECT
    year(bk_bookingdate)                                         AS booking_year,
    round(sum(bk_totalprice), 2)                                 AS annual_net_revenue,
    count(DISTINCT bk_bookingid)                                 AS distinct_booking,
    count(DISTINCT bk_passengerid)                               AS active_passengers,
    round(sum(bk_totalprice) / count(DISTINCT bk_bookingid), 2)  AS average_booking_value
FROM {catalog}.silver.booking
WHERE bk_status <> 'C'
GROUP BY year(bk_bookingdate)
ORDER BY booking_year
""")

![job_screenshot_1792499883069.png](./job_screenshot_1792499883069.png "job_screenshot_1792499883069.png")